In [8]:
!pip install pypdf sentence-transformers chromadb python-dotenv requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [9]:
from __future__ import annotations

import json
import logging
import os
import re
import shutil
import time
from dataclasses import dataclass
from io import BytesIO
from pathlib import Path
from typing import Any, Callable, Dict, List, Sequence

import numpy as np
from dotenv import load_dotenv
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer



In [10]:
try:
    import chromadb
    #from chromadb.config import Settings
except ImportError as exc:  # pragma: no cover - import guard
    raise ImportError(
        "The 'chromadb' package is required. Install it with 'pip install chromadb'."
    ) from exc

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("amazon_rag")

In [12]:
from openai import OpenAI

OPENAI_API_KEY="copy here api key"
openai_client = OpenAI(api_key=OPENAI_API_KEY)

In [13]:
CHROMA_DIR = Path("chroma_store11")
CHROMA_DIR.mkdir(exist_ok=True)

COMPANY_NAME = "Amazon"
TOP_K = 3

PDF_DIR = Path("amazon_annual_report")


In [14]:
PDF_DIR

PosixPath('amazon_annual_report')

In [15]:
PDF_DIR.exists()

True

In [16]:
CHUNK_SIZE = 250      # words, for sliding window
CHUNK_OVERLAP = 40    # words

In [17]:
# Step 1: Extract text from PDFs
# ---------------------------------------------------------------------------

def extract_pages(pdf_path):
    """Return a list of page texts for one PDF."""
    reader = PdfReader(str(pdf_path))
    return [page.extract_text() or "" for page in reader.pages]


In [18]:
# ---------------------------------------------------------------------------
# Step 2: Chunking strategies - fixed
# ---------------------------------------------------------------------------

def sliding_window_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word windows."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks


def paragraph_chunk(text, max_chars=400):
    """Split text into paragraphs, merging small ones up to max_chars."""
    parts = [p.strip() for p in re.split(r"\n{2,}|\. ", text) if p.strip()]
    chunks = []
    buffer = ""
    for part in parts:
        candidate = (buffer + " " + part).strip()
        if len(candidate) <= max_chars:
            buffer = candidate
        else:
            if buffer:
                chunks.append(buffer)
            buffer = part
    if buffer:
        chunks.append(buffer)
    return chunks

In [19]:
# Step 3: Build the corpus (list of simple dicts) - 100 pages  -> 48th page - 3 chunks
# ---------------------------------------------------------------------------

def build_corpus(pdf_dir):
    corpus = []
    pdf_files = sorted(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF(s) in {pdf_dir.resolve()}")

    for pdf_path in pdf_files:
        report_name = pdf_path.stem
        pages = extract_pages(pdf_path)
        print(f"  {report_name}: {len(pages)} pages")

        for page_num, page_text in enumerate(pages, start=1):
            page_text = page_text.strip()
            if not page_text:
                continue

            for method_name, chunker in [
                ("sliding_window", sliding_window_chunk),
                ("paragraph", paragraph_chunk),
            ]:
                for i, chunk_text in enumerate(chunker(page_text)):
                    if not chunk_text.strip():
                        continue
                    corpus.append({
                        "id": f"{report_name}-{method_name}-p{page_num}-{i}",
                        "text": chunk_text,
                        "page": page_num,
                        "method": method_name,
                        "report": report_name,
                        "company": COMPANY_NAME,
                    })

    print(f"Total chunks: {len(corpus)}")
    return corpus

In [20]:
#Step 4: Embeddings
# ---------------------------------------------------------------------------

def embed_texts(model, texts):
    return model.encode(texts, normalize_embeddings=True, convert_to_numpy=True,
                         show_progress_bar=True).astype(np.float32)



In [21]:
# ---------------------------------------------------------------------------
# Step 5: Retrieval - plain numpy cosine similarity
# ---------------------------------------------------------------------------

def cosine_retrieval(query_vec, chunk_vecs, corpus, top_k=TOP_K):
    scores = chunk_vecs @ query_vec
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{**corpus[i], "score": float(scores[i])} for i in top_idx]

In [22]:
#---------------------------------------------------------------------------
# Step 6: Retrieval - ChromaDB HNSW index
# ---------------------------------------------------------------------------

def build_chroma_collection(corpus, chunk_vecs, persist_dir=CHROMA_DIR):
    if persist_dir.exists():
        shutil.rmtree(persist_dir)
    client = chromadb.PersistentClient(path=str(persist_dir))
    collection = client.create_collection(name="amazon_reports", metadata={"hnsw:space": "cosine"})

    collection.add(
        ids=[c["id"] for c in corpus],
        embeddings=chunk_vecs.tolist(),
        documents=[c["text"] for c in corpus],
        metadatas=[{"page": c["page"], "report": c["report"], "method": c["method"]} for c in corpus],
    )
    return collection


def hnsw_retrieval(collection, query_vec, top_k=TOP_K):
    res = collection.query(query_embeddings=[query_vec.tolist()], n_results=top_k,
                            include=["documents", "metadatas", "distances"])
    results = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        results.append({"text": doc, "score": 1 - dist, **meta})
    return results



In [23]:
# Step 7: Orchestration
# ---------------------------------------------------------------------------

def setup_pipeline():
    corpus = build_corpus(PDF_DIR)
    if not corpus:
        raise RuntimeError(f"No text extracted. Check that PDFs exist in {PDF_DIR.resolve()}")

    print("Loading embedding model...")
    model = SentenceTransformer("BAAI/bge-small-en-v1.5")

    print("Embedding chunks...")
    chunk_vecs = embed_texts(model, [c["text"] for c in corpus])

    print("Building Chroma HNSW index...")
    collection = build_chroma_collection(corpus, chunk_vecs)

    return {"model": model, "corpus": corpus, "chunk_vecs": chunk_vecs, "collection": collection}

In [24]:
def setup_pipeline_with_llm():
    corpus = build_corpus(PDF_DIR)
    if not corpus:
        raise RuntimeError(f"No text extracted. Check that PDFs exist in {PDF_DIR.resolve()}")

    print("Loading embedding model...")
    model = SentenceTransformer("BAAI/bge-small-en-v1.5")

    print("Embedding chunks...")
    chunk_vecs = embed_texts(model, [c["text"] for c in corpus])

    print("Building Chroma HNSW index...")
    collection = build_chroma_collection(corpus, chunk_vecs)

    # The openai_client is now a global variable, so we can directly access it.
    return {"model": model, "corpus": corpus, "chunk_vecs": chunk_vecs, "collection": collection, "openai_client": openai_client}

In [25]:
def ask(state, query, top_k=TOP_K):
    model = state["model"]
    query_vec = np.asarray(model.encode(query, normalize_embeddings=True), dtype=np.float32)

    t0 = time.perf_counter()
    cosine_results = cosine_retrieval(query_vec, state["chunk_vecs"], state["corpus"], top_k)
    t1 = time.perf_counter()
    hnsw_results = hnsw_retrieval(state["collection"], query_vec, top_k)
    t2 = time.perf_counter()

    print(f"\nQuery: {query}")
    print(f"\n[Cosine similarity]  (latency {t1 - t0:.4f}s)")
    for r in cosine_results:
        print(f"  score={r['score']:.4f} | {r['report']} p{r['page']} ({r['method']})")
        print(f"  -> {r['text'][:200]}...")

    print(f"\n[Chroma HNSW]  (latency {t2 - t1:.4f}s)")
    for r in hnsw_results:
        print(f"  score={r['score']:.4f} | {r['report']} p{r['page']} ({r['method']})")
        print(f"  -> {r['text'][:200]}...")

    return {"cosine": cosine_results, "hnsw": hnsw_results}

In [26]:
def ask_with_llm(state, query, top_k: int = 3, model_name="gpt-4o-mini"):
    model = state["model"]
    openai_client = state["openai_client"]

    query_vec = np.asarray(model.encode(query, normalize_embeddings=True), dtype=np.float32)

    t0 = time.perf_counter()
    cosine_results = cosine_retrieval(query_vec, state["chunk_vecs"], state["corpus"], top_k)
    t1 = time.perf_counter()
    hnsw_results = hnsw_retrieval(state["collection"], query_vec, top_k)
    t2 = time.perf_counter()

    # Combine retrieved documents from both methods into a single context string
    combined_results = cosine_results + hnsw_results
    context = "\n\n".join([r["text"] for r in combined_results])

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant that answers questions based on the provided context.\n"
                "If the answer is not available in the context, clearly state that.\n"
                "Keep your answers concise and accurate."
            ),
        },
        {
            "role": "user",
            "content": f"Context: {context}\n\nQuestion: {query}",
        },
    ]

    print(f"\nQuery: {query}")
    print(f"\n[Cosine similarity retrieval]  (latency {t1 - t0:.4f}s)")
    for r in cosine_results:
        print(f"  score={r['score']:.4f} | {r['report']} p{r['page']} ({r['method']})")
        print(f"  -> {r['text'][:100]}...")

    print(f"\n[Chroma HNSW retrieval]  (latency {t2 - t1:.4f}s)")
    for r in hnsw_results:
        print(f"  score={r['score']:.4f} | {r['report']} p{r['page']} ({r['method']})")
        print(f"  -> {r['text'][:100]}...")

    try:
        response = openai_client.chat.completions.create(
            model=model_name,
            messages=messages,
            temperature=0.0 # Make the output more deterministic
        )
        llm_answer = response.choices[0].message.content
        print(f"\n[LLM Answer]\n{llm_answer}")
        return llm_answer
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        return ""

In [29]:
#if __name__ == "__main__":
#    state = setup_pipeline()
#    ask(state, "what is the leased square space and owned aquare sapce for amazon on North America.")

In [28]:
if __name__ == "__main__":
    # Use the new setup function that includes the OpenAI client
    state_llm = setup_pipeline_with_llm()
    ask_with_llm(state_llm, "what is the leased square space and owned square space for amazon on North America.")

Found 1 PDF(s) in /content/amazon_annual_report
  Amazon-2025-Annual-Report: 92 pages
Total chunks: 1135
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding chunks...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Building Chroma HNSW index...

Query: what is the leased square space and owned square space for amazon on North America.

[Cosine similarity retrieval]  (latency 0.0010s)
  score=0.8041 | Amazon-2025-Annual-Report p29 (sliding_window)
  -> Item 2. Properties As of December 31, 2025, we operated the following facilities (in thousands): Des...
  score=0.7878 | Amazon-2025-Annual-Report p29 (paragraph)
  -> Properties
As of December 31, 2025, we operated the following facilities (in thousands):
Description...
  score=0.7525 | Amazon-2025-Annual-Report p29 (paragraph)
  -> Segment
Leased Square 
Footage (1)
Owned Square 
Footage (1)
North America 481,153 25,437
Internatio...

[Chroma HNSW retrieval]  (latency 0.0056s)
  score=0.8041 | Amazon-2025-Annual-Report p29 (sliding_window)
  -> Item 2. Properties As of December 31, 2025, we operated the following facilities (in thousands): Des...
  score=0.7878 | Amazon-2025-Annual-Report p29 (paragraph)
  -> Properties
As of December 31, 2025, we